In [1]:
## RAG APPLICATION USING TYPESENSE

import typesense


In [ ]:
client = typesense.Client({
    'nodes' : [{
        'host' : '1i38bw02kcx7aepgp-1.a2.typesense.net',
        'port' : '443',
        'protocol' : 'https'

    }],
    'key deleted
    'connection_timeout_seconds':2
})




In [3]:
client

In [4]:
books_schema ={
    'name' : 'books',
    'fields': [
        {'name' : 'title','type':'string'},
        {'name': 'authors','type':'string[]','facet':True},
        {'name': 'publication_year','type':'int32','facet':True},
        {'name': 'ratings_count','type':'int32'},
        {'name': 'average_rating','type':'float'}
    ],
    'default_sorting_field': 'ratings_count'
}

In [5]:
print(client.collections.create(books_schema))

ObjectAlreadyExists: [Errno 409] A collection with name `books` already exists.

In [6]:
with open('books.jsonl','r',encoding='utf-8') as jsonl_file:
    data = jsonl_file.read()
    client.collections['books'].documents.import_(data)

In [7]:
search_parameters= {
    'q': "harry potter",
    'query_by' : "title,authors",
    'sort_by' : "ratings_count:desc"
}

In [8]:
client.collections['books'].documents.search(search_parameters)

{'facet_counts': [],
 'found': 17,
 'hits': [{'document': {'authors': ['J.K. Rowling', ' Mary GrandPré'],
    'average_rating': 4.44,
    'id': '2',
    'image_url': 'https://images.gr-assets.com/books/1474154022m/3.jpg',
    'publication_year': 1997,
    'ratings_count': 4602479,
    'title': "Harry Potter and the Philosopher's Stone"},
   'highlight': {'title': {'matched_tokens': ['Harry', 'Potter'],
     'snippet': "<mark>Harry</mark> <mark>Potter</mark> and the Philosopher's Stone"}},
   'highlights': [{'field': 'title',
     'matched_tokens': ['Harry', 'Potter'],
     'snippet': "<mark>Harry</mark> <mark>Potter</mark> and the Philosopher's Stone"}],
   'text_match': 1157451471441102969,
   'text_match_info': {'best_field_score': '2211897868289',
    'best_field_weight': 15,
    'fields_matched': 1,
    'num_tokens_dropped': 0,
    'score': '1157451471441102969',
    'tokens_matched': 2,
    'typo_prefix_score': 0}},
  {'document': {'authors': ['J.K. Rowling', ' Mary GrandPré', ' R

In [1]:
## langchain + typesense + groq llm + rag
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import Typesense
from langchain_text_splitters import CharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq

C:\Users\11c16\AppData\Local\Temp\ipykernel_3712\2739331114.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [10]:
!pip install langchain-huggingface

In [ ]:
import os

key deleted

In [12]:
loader = TextLoader("test.txt")
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=1000,chunk_overlap=100)
docs = text_splitter.split_documents(documents)
embeddings = HuggingFaceEmbeddings()

In [ ]:
docsearch=Typesense.from_documents(
    docs,
    embeddings,
    typesense_client_params={
        keys deleted.
    },
) 

In [14]:
query ="Explain the complete architecture of LangChain and describe how its core components—including prompt templates, document loaders, text splitters, embeddings, vector databases, retrievers, chains, memory, agents, and tools—work together to build a Retrieval-Augmented Generation (RAG) application"
found_docs = docsearch.similarity_search(query)
print(found_docs[0].page_content)

The closest matches become retrieval results.

---

## Retrieval-Augmented Generation

Perhaps the most important LangChain application is Retrieval-Augmented Generation (RAG).

Traditional language models only know information from training.

RAG supplements the model with external knowledge.

Pipeline:

Documents

↓

Chunking

↓

Embeddings

↓

Vector Database

↓

User Query

↓

Query Embedding

↓

Similarity Search

↓

Relevant Chunks

↓

Prompt

↓

LLM

↓

Answer

This approach greatly reduces hallucinations because responses are grounded in retrieved evidence.

RAG is widely used in:

Customer support

Enterprise search

Medical assistants

Legal assistants

Research tools

Internal company chatbots

Document Q&A

Knowledge bases

---

## Retrievers

Retrievers determine which documents should be passed to the language model.

LangChain provides multiple retrieval strategies.

Examples include:

Similarity Search

MMR (Maximum Marginal Relevance)

Self Query Retriever


In [15]:
retriever = docsearch.as_retriever()
retriever

VectorStoreRetriever(tags=['Typesense', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.typesense.Typesense object at 0x00000245A7EDF410>, search_kwargs={})

In [16]:
retriever.invoke(query)[0].page_content

'The closest matches become retrieval results.\n\n---\n\n## Retrieval-Augmented Generation\n\nPerhaps the most important LangChain application is Retrieval-Augmented Generation (RAG).\n\nTraditional language models only know information from training.\n\nRAG supplements the model with external knowledge.\n\nPipeline:\n\nDocuments\n\n↓\n\nChunking\n\n↓\n\nEmbeddings\n\n↓\n\nVector Database\n\n↓\n\nUser Query\n\n↓\n\nQuery Embedding\n\n↓\n\nSimilarity Search\n\n↓\n\nRelevant Chunks\n\n↓\n\nPrompt\n\n↓\n\nLLM\n\n↓\n\nAnswer\n\nThis approach greatly reduces hallucinations because responses are grounded in retrieved evidence.\n\nRAG is widely used in:\n\nCustomer support\n\nEnterprise search\n\nMedical assistants\n\nLegal assistants\n\nResearch tools\n\nInternal company chatbots\n\nDocument Q&A\n\nKnowledge bases\n\n---\n\n## Retrievers\n\nRetrievers determine which documents should be passed to the language model.\n\nLangChain provides multiple retrieval strategies.\n\nExamples include:\n\

In [17]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

In [18]:
retriever = docsearch.as_retriever(
    search_kwargs={"k": 3}
)

In [19]:
import langchain
print(langchain.__version__)

1.3.11


In [20]:
query = "Explain complete process of langchain"


In [21]:

retrieved_docs = retriever.invoke(query)

In [22]:
context = "\n\n".join(
    [doc.page_content for doc in retrieved_docs]
)

In [23]:
prompt = f"""
You are a helpful AI assistant.

Answer the user's question ONLY using the context provided below.
If the answer is not available in the context, say
"I don't have enough information."

{context}

Question:
{query}

Answer:
"""

In [24]:
response = llm.invoke(prompt)

In [25]:
print("RETRIEVED CHUNKS")
for i, doc in enumerate(retrieved_docs, start=1):
    print(f"\n---------- Chunk {i} ----------\n")
    print(doc.page_content)

RETRIEVED CHUNKS

---------- Chunk 1 ----------

Each component performs one responsibility extremely well.

These components can then be connected together into sophisticated AI pipelines.

---

## Architecture

A typical LangChain application follows this flow:

User Question

↓

Prompt Template

↓

LLM

↓

Output Parser

↓

Final Response

For Retrieval-Augmented Generation:

User Question

↓

Retriever

↓

Vector Database

↓

Relevant Documents

↓

Prompt Template

↓

Large Language Model

↓

Answer

For AI Agents:

User

↓

Agent

↓

Reasoning

↓

Tool Selection

↓

API / Database / Calculator / Search Engine

↓

Observation

↓

LLM

↓

Final Answer

This modular architecture makes LangChain extremely flexible.

---

## Models

The language model is the heart of every LangChain application.

LangChain supports virtually every popular provider.

Examples include:

* OpenAI GPT
* Anthropic Claude
* Google Gemini
* Groq
* Ollama
* Hugging Face
* Mistral AI
* Cohere
* Together AI
* Az

In [26]:
print("aNswer")
print(response.content)

aNswer
The complete process of LangChain can be explained in three main flows:

1. **Typical LangChain Application**:
   - User Question
   - Prompt Template
   - LLM (Large Language Model)
   - Output Parser
   - Final Response

2. **Retrieval-Augmented Generation**:
   - User Question
   - Retriever
   - Vector Database
   - Relevant Documents
   - Prompt Template
   - Large Language Model
   - Answer

3. **AI Agents**:
   - User
   - Agent
   - Reasoning
   - Tool Selection
   - API / Database / Calculator / Search Engine
   - Observation
   - LLM
   - Final Answer

This process showcases the modular architecture of LangChain, which makes it extremely flexible. Each component performs one responsibility extremely well and can be connected together into sophisticated AI pipelines.


In [27]:
retriever = docsearch.as_retriever(
    search_kwargs={"k": 10}
)

retrieved_docs = retriever.invoke(query)

context = "\n\n".join(
    [doc.page_content for doc in retrieved_docs]
)

prompt = f"""
You are a helpful AI assistant.

Answer the user's question ONLY using the context provided below.
If the answer is not available in the context, say
"I don't have enough information."

{context}

Question:
{query}

Answer:
"""


In [28]:
response = llm.invoke(prompt)


In [29]:
print("RETRIEVED CHUNKS")
for i, doc in enumerate(retrieved_docs, start=1):
    print(f"\n---------- Chunk {i} ----------\n")
    print(doc.page_content)

RETRIEVED CHUNKS

---------- Chunk 1 ----------

Each component performs one responsibility extremely well.

These components can then be connected together into sophisticated AI pipelines.

---

## Architecture

A typical LangChain application follows this flow:

User Question

↓

Prompt Template

↓

LLM

↓

Output Parser

↓

Final Response

For Retrieval-Augmented Generation:

User Question

↓

Retriever

↓

Vector Database

↓

Relevant Documents

↓

Prompt Template

↓

Large Language Model

↓

Answer

For AI Agents:

User

↓

Agent

↓

Reasoning

↓

Tool Selection

↓

API / Database / Calculator / Search Engine

↓

Observation

↓

LLM

↓

Final Answer

This modular architecture makes LangChain extremely flexible.

---

## Models

The language model is the heart of every LangChain application.

LangChain supports virtually every popular provider.

Examples include:

* OpenAI GPT
* Anthropic Claude
* Google Gemini
* Groq
* Ollama
* Hugging Face
* Mistral AI
* Cohere
* Together AI
* Az

In [30]:
print("aNswer")
print(response.content)

aNswer
The complete process of LangChain involves several components and steps. Here's an overview:

1. **User Question**: The user asks a question or provides input.
2. **Prompt Template**: The input is processed using a prompt template to create a prompt for the language model.
3. **LLM (Large Language Model)**: The prompt is sent to the LLM, which generates a response.
4. **Output Parser**: The response from the LLM is parsed to extract relevant information.
5. **Final Response**: The parsed response is then provided as the final answer to the user.

For **Retrieval-Augmented Generation**:

1. **User Question**: The user asks a question or provides input.
2. **Retriever**: The input is used to retrieve relevant documents or information from a database or knowledge base.
3. **Vector Database**: The retrieved documents are stored in a vector database.
4. **Relevant Documents**: The relevant documents are extracted from the vector database.
5. **Prompt Template**: The extracted documen